# Stg Layer — video_transcripts (Gemini API, free tier, cached)

Lấy transcript tiếng Việt của mỗi video (`${gold}.dim_video`) bằng **Gemini API**: gửi thẳng URL YouTube, Google tự đọc video (Databricks không gọi youtube.com nên không bị chặn IP) → Silver `${silver}.stg_video_transcripts`, 1 row / video.

**Flow:**
1. Video **chưa có transcript** trong cache, ngắn hơn `MAX_VIDEO_SECONDS`
2. Kiểm tra model (`model` widget) còn dùng được với key này; không thì fail ngay và in danh sách model Flash hiện có
3. Gọi Gemini trong **daily cap**: tổng độ dài video đã transcribe thành công ≤ `DAILY_VIDEO_SECONDS` mỗi ngày (giờ VN, cộng dồn mọi run trong ngày từ `stg_video_transcripts`), tối đa `max_videos` **lần gọi** mỗi run; phần còn lại chờ hôm sau
4. MERGE vào cache (insert-only theo `video_id`). Lỗi HTTP **không** cache, run sau thử lại; `MAX_CONSECUTIVE_ERRORS` lỗi liên tiếp thì dừng run.
5. Log mỗi run vào `stg_video_transcript_calls` (chỉ để theo dõi).

**Free tier only:** API key phải tạo trong một GCP project **không gắn billing** (không dùng `YouTubeapiv1`, project đó có billing → thành paid tier). Không có billing thì vượt hạn mức chỉ trả HTTP 429, không tính tiền. Free tier giới hạn ~8 giờ video YouTube / ngày; cap ở đây là 7 giờ. Trên 429 run dừng gọi API nhưng **không** fail. Key bị từ chối (400 / 403) thì fail task.

**Lưu ý:** chỉ video public. Free tier: Google có thể dùng dữ liệu gửi lên để cải thiện sản phẩm.

**Secret:** `gemini/api_key`. Dùng `requests` (REST), không cần cài SDK.

In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
dbutils.widgets.text("max_videos", "5")  # videos per run (5 = test run)
MAX_VIDEOS = int(dbutils.widgets.get("max_videos"))
dbutils.widgets.text("min_seconds", "0")  # e.g. 300 to test a longer review instead of shorts
MIN_VIDEO_SECONDS = int(dbutils.widgets.get("min_seconds"))
dbutils.widgets.text("video_id", "")  # optional: transcribe only this video (test)
ONLY_VIDEO_ID = dbutils.widgets.get("video_id").strip()
# gemini-2.5-flash is no longer offered to new keys (HTTP 404); the preflight cell checks this name
dbutils.widgets.text("model", "gemini-3.8-flash")
MODEL = dbutils.widgets.get("model")

VIDEOS_TABLE = f"{GOLD}.dim_video"
TRANSCRIPTS_TABLE = f"{SILVER}.stg_video_transcripts"
CALLS_TABLE = f"{SILVER}.stg_video_transcript_calls"  # video seconds sent per run, for the daily cap
DAILY_VIDEO_SECONDS = 7 * 3600   # free tier allows ~8 h of YouTube video per day
MAX_VIDEO_SECONDS = 30 * 60      # longer videos would exceed the free-tier tokens per minute
SLEEP_SECONDS = 10               # free-tier requests per minute
MAX_CONSECUTIVE_ERRORS = 3
SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
NO_SPEECH = "[không có lời nói]"
PROMPT = (
    "Chép lại nguyên văn lời nói (người nói, thuyết minh) tiếng Việt trong video này. "
    "Bỏ qua lời bài hát và nhạc nền. "
    "Mỗi đoạn một dòng, mở đầu bằng mốc thời gian [mm:ss]. "
    "Chỉ trả về phần chép lại: không câu mở đầu, không tóm tắt, không thêm lời bình. "
    f"Nếu video không có lời nói, chỉ trả về đúng: {NO_SPEECH}"
)

print(f"Videos source: {VIDEOS_TABLE}")
print(f"Model: {MODEL}")
print(f"Transcript cache: {TRANSCRIPTS_TABLE} · max {MAX_VIDEOS} calls per run, {DAILY_VIDEO_SECONDS // 3600} h video per day")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TRANSCRIPTS_TABLE} (
        video_id         STRING,
        transcript       STRING,
        video_seconds    INT,
        prompt_tokens    INT,
        output_tokens    INT,
        model_name       STRING,
        transcribed_at   TIMESTAMP
    )
    USING DELTA
    COMMENT 'Gemini transcript per YouTube video (1 row per video, insert-only). transcript = [không có lời nói] when the video has no speech, NULL when the model returned nothing.'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CALLS_TABLE} (
        run_ts        TIMESTAMP,
        call_date     DATE,
        api_calls     INT,
        video_seconds INT,
        errors        INT
    )
    USING DELTA
    COMMENT 'Gemini calls per run of stg_video_transcripts (daily free-tier cap).'
""")

# --- Video seconds transcribed today (Vietnam day) by earlier runs; failed calls do not count ---
used_today = spark.sql(f"""
    SELECT coalesce(sum(video_seconds), 0) AS s FROM {TRANSCRIPTS_TABLE}
    WHERE to_date(from_utc_timestamp(transcribed_at, '{SNAPSHOT_TZ}'))
        = to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}'))
""").collect()[0]["s"]

# --- Videos without a transcript yet, shortest first (more videos per day) ---
todo = spark.sql(f"""
    SELECT v.video_id, v.duration_seconds
    FROM {VIDEOS_TABLE} v
    LEFT ANTI JOIN {TRANSCRIPTS_TABLE} t ON t.video_id = v.video_id
    WHERE v.duration_seconds > greatest({MIN_VIDEO_SECONDS}, 0) AND v.duration_seconds <= {MAX_VIDEO_SECONDS}
      AND ('{ONLY_VIDEO_ID}' = '' OR v.video_id = '{ONLY_VIDEO_ID}')
    ORDER BY v.duration_seconds, v.video_id
""").collect()
print(f"Videos without transcript: {len(todo)} · video time already used today: {used_today / 3600:.2f} h")

In [ ]:
# --- Call Gemini with the YouTube URL ---
import time

import requests

API_KEY = dbutils.secrets.get("gemini", "api_key")
BASE = "https://generativelanguage.googleapis.com/v1beta"
URL = f"{BASE}/models/{MODEL}:generateContent"

# --- Preflight: fail fast when the model name is not offered to this key ---
check = requests.get(f"{BASE}/models/{MODEL}", headers={"x-goog-api-key": API_KEY}, timeout=30)
if check.status_code != 200:
    listed = requests.get(f"{BASE}/models", headers={"x-goog-api-key": API_KEY},
                          params={"pageSize": 1000}, timeout=30).json().get("models", [])
    flash = sorted(m["name"].removeprefix("models/") for m in listed
                   if "flash" in m["name"] and "generateContent" in m.get("supportedGenerationMethods", []))
    raise RuntimeError(f"Model {MODEL} not available (HTTP {check.status_code}). "
                       f"Set the model widget to one of: {flash}")


def transcribe(video_id):
    body = {
        "contents": [{"parts": [
            {"file_data": {"file_uri": f"https://www.youtube.com/watch?v={video_id}"}},
            {"text": PROMPT},
        ]}],
        # Low resolution: ~3x fewer tokens per second of video; the audio is what matters here
        "generationConfig": {"mediaResolution": "MEDIA_RESOLUTION_LOW", "temperature": 0},
    }
    resp = requests.post(URL, headers={"x-goog-api-key": API_KEY}, json=body, timeout=600)
    if resp.status_code != 200:
        return None, resp.status_code, resp.text[:300]
    data = resp.json()
    parts = (data.get("candidates") or [{}])[0].get("content", {}).get("parts", [])
    text = "".join(p.get("text", "") for p in parts).strip()
    # Drop a "Dưới đây là ..." preamble line if the model still adds one
    lines = text.splitlines()
    if lines and not lines[0].lstrip().startswith("[") and lines[0].rstrip().endswith(":"):
        text = "\n".join(lines[1:]).strip()
    # NO_SPEECH is stored as is (final answer); NULL means the model returned nothing
    usage = data.get("usageMetadata", {})
    return {"transcript": text or None, "prompt_tokens": usage.get("promptTokenCount"),
            "output_tokens": usage.get("candidatesTokenCount")}, 200, None


results, api_calls, sent_seconds, errors, consecutive_errors = [], 0, 0, 0, 0
for row in todo:
    if api_calls >= MAX_VIDEOS:
        break
    if used_today + sent_seconds + row["duration_seconds"] > DAILY_VIDEO_SECONDS:
        print("Daily video cap reached; the rest waits for tomorrow")
        break
    if api_calls:
        time.sleep(SLEEP_SECONDS)
    out, status, err = transcribe(row["video_id"])
    api_calls += 1
    if status == 429:
        print(f"HTTP 429 (free-tier limit) at {row['video_id']}; stopping, the rest waits for the next run")
        break
    if status in (400, 401, 403) and "API key" in (err or ""):
        raise RuntimeError(f"Gemini rejected the API key: {err}")
    if out is None:
        errors += 1
        consecutive_errors += 1
        print(f"❌ {row['video_id']}: HTTP {status} {err}")
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            print(f"{consecutive_errors} errors in a row; stopping, the rest waits for the next run")
            break
        continue
    consecutive_errors = 0
    sent_seconds += row["duration_seconds"]
    results.append({"video_id": row["video_id"], "video_seconds": row["duration_seconds"], **out})
    print(f"✅ {row['video_id']} ({row['duration_seconds'] // 60} min): "
          f"{len(out['transcript'] or '')} chars, {out['prompt_tokens']} prompt tokens")

print(f"API calls: {api_calls} · transcripts: {len(results)} · errors: {errors} · video sent: {sent_seconds / 60:.0f} min")

In [ ]:
# --- MERGE new transcripts into the cache (insert-only) and log the run ---
from pyspark.sql.functions import current_timestamp, lit

if results:
    spark.createDataFrame(
        results,
        "video_id STRING, transcript STRING, video_seconds INT, prompt_tokens INT, output_tokens INT",
    ).withColumn("model_name", lit(MODEL)).withColumn("transcribed_at", current_timestamp()) \
     .createOrReplaceTempView("transcripts_src")
    spark.sql(f"""
        MERGE INTO {TRANSCRIPTS_TABLE} AS t
        USING transcripts_src AS s
        ON t.video_id = s.video_id
        WHEN NOT MATCHED THEN INSERT *
    """)
spark.sql(f"""
    INSERT INTO {CALLS_TABLE}
    SELECT current_timestamp(), to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}')),
           {api_calls}, {sent_seconds}, {errors}
""")
print(f"✅ Merged {len(results)} transcripts into {TRANSCRIPTS_TABLE}")

In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT video_id) AS n_videos, COUNT(transcript) AS with_text,
           round(sum(video_seconds) / 3600, 2) AS hours, sum(prompt_tokens) AS prompt_tokens
    FROM {TRANSCRIPTS_TABLE}
""").collect()[0]
print(f"📊 {TRANSCRIPTS_TABLE}: {stats['n']} videos ({stats['with_text']} with text, {stats['hours']} h video, "
      f"{stats['prompt_tokens']} prompt tokens)")
print(f"📊 Grain check (video_id): {stats['n']} rows vs {stats['n_videos']} distinct "
      + ("✅" if stats["n"] == stats["n_videos"] else "❌ duplicates"))
display(spark.sql(f"""
    SELECT video_id, video_seconds, left(transcript, 300) AS transcript_start
    FROM {TRANSCRIPTS_TABLE} ORDER BY transcribed_at DESC LIMIT 10
"""))
print("\n--- stg_video_transcripts complete ---")